In [1]:
!pip install pandas openpyxl numpy

# Lihat Preview Pilot Testing

In [4]:
import pandas as pd

nama_file_csv = 'dataset_properti_raw.csv'

try:
    df = pd.read_csv(nama_file_csv)
    
    display(df.tail())

except FileNotFoundError:
    print(f"Error: File '{nama_file_csv}' tidak ditemukan.")
    print("Pastikan file CSV sudah di-upload ke folder yang sama dengan notebook ini.")

,ad_id,title,price,lat,lon,Provinsi,Kota/kabupaten,Kecamatan,Luas_bangunan,Luas_tanah,Kamar_tidur,Kamar_Mandi,parameter-external_source_url,Tipe,Fasilitas,Lantai,description
7239,945097179,Sutorejo Surabaya,1.999000e+09,-7.267,112.744,Jawa Timur,Surabaya Kota,Gubeng,160.0,90.0,3,4,https://www.lamudi.co.id/properti/41032-73-fe6...,rumah,garasi,2.0,"Sutorejo Surabaya, Surabaya Kota, Surabaya, Ja..."
7240,945097172,"Araya 2 dkt merr, kertajaya, dharmahusada, manyar",3.900000e+09,-7.301,112.781,Jawa Timur,Surabaya Kota,Mulyorejo,255.0,135.0,4,3,https://www.lamudi.co.id/properti/41032-73-b4b...,rumah,garasi,3.0,"Araya 2 dkt merr, kertajaya, dharmahusada, man..."
7241,945097164,pakuwon city,3.900000e+09,-7.276,112.805,Jawa Timur,Surabaya Kota,Sukolilo,240.0,160.0,4,4,https://www.lamudi.co.id/properti/41032-73-2e2...,rumah,garasi,2.0,"pakuwon city, Pakuwon City, Surabaya, Jawa Tim..."
7242,945097153,Sutorejo Surabaya,1.999000e+09,-7.267,112.744,Jawa Timur,Surabaya Kota,Gubeng,160.0,90.0,3,4,https://www.lamudi.co.id/properti/41032-73-215...,rumah,garasi,2.0,"Sutorejo Surabaya, Surabaya Kota, Surabaya, Ja..."
7243,945097151,DHARMAHUSADA MAS,8.500000e+09,-7.268,112.774,Jawa Timur,Surabaya Kota,Mulyorejo,575.0,450.0,5,6,https://www.lamudi.co.id/properti/41032-73-709...,rumah,garasi,2.0,"DHARMAHUSADA MAS, Dharma Husada, Surabaya, Jaw..."


# Konversi ke Excel

In [6]:
nama_file_excel = 'pilot_testing_enriched.xlsx'

try:
    df.to_excel(nama_file_excel, index=False, engine='openpyxl')
    
except Exception as e:
    print(f"Terjadi kesalahan saat menyimpan file: {e}")

# Standardisasi Istilah Fasilitas

In [ ]:
synonym_map = {
    "garden": "Taman",
    "garasi": "Parkiran mobil",
    "carport": "Parkiran mobil",
    "swimmingpool": "Kolam renang",
    "gordyn": "Gorden",
    "pam": "Air",
    "waterheater": "Pemanas air",
    "refrigerator": "Kulkas",
    "stove": "Kompor",
    "fireextenguisher": "APAR (Alat Pemadam Api Ringan)",
    "ac": "Pendingin ruangan (AC)",
    "pendingin ruangan (ac)": "Pendingin ruangan (AC)",
    "keamanan": "Sistem Keamanan",
    "keamanan 24 jam": "Sistem Keamanan",
    "telephone": "Telepon"
}

def process_facilities(row):
    # Gabungkan kolom "Fasilitas", "Fasilitas_Indoor", dan "Karakteristik_bangunan"
    f1 = str(row['Fasilitas']) if pd.notna(row['Fasilitas']) else ""
    f2 = str(row['Fasilitas_Indoor']) if pd.notna(row['Fasilitas_Indoor']) else ""
    f3 = str(row['Karakteristik_bangunan']) if pd.notna(row['Karakteristik_bangunan']) else ""
    
    raw_combined = f"{f1},{f2},{f3}"
    items = [item.strip().lower() for item in raw_combined.split(',')]
    
    standardized_items = set()
    for item in items:
        if item == "" or item == "nan":
            continue
        mapped_value = synonym_map.get(item, item.title())
        standardized_items.add(mapped_value)
        
    return ", ".join(sorted(standardized_items))

# Eksekusi Standardisasi

In [16]:
df['Facilities'] = df.apply(process_facilities, axis=1)

df = df.drop(columns=['Fasilitas', 'Fasilitas_Indoor', 'Karakteristik_bangunan'])

df[['title', 'Facilities']].head(5)

,title,Facilities
0,"Rumah Baru 2 Lantai, Bagus dan Strategis di Wi...","Air, Listrik, Pagar Penuh, Parkiran mobil, Tan..."
1,Lantai 2‼️155 jt • 2 BR Termurah Apartemen Pun...,Sebagian Perabotan
2,Diamond Hill Citraland,"Parkiran mobil, Tanpa Perabotan"
3,"‼️BARU GRESS 2 UNIT‼️ RUMAH WISMA MUKTI, SEMAL...",Tanpa Perabotan
4,RUMAH MEWAH SIAP HUNI 2 LT DHARMAHUSADA SURABA...,"Listrik, Tanpa Perabotan"


# Simpan Update Pilot Testing

In [17]:
df.to_csv("pilot_testing_cleaned.csv", index=False)